# 03 — Descarga masiva de imágenes (65k)

Descarga todas las imágenes de train + val a Google Drive **sin cargar el modelo ni entrenar**.
Correr esta sesión una sola vez antes del sprint de entrenamiento completo.

| Split | Ejemplos | Imgs estimadas |
|-------|----------|----------------|
| train | 65,964   | ~330k          |
| val   | 23,773   | ~119k          |

**Tiempo estimado:** 3-6 horas (depende de la CDN y la conexión de Drive).  
**No requiere GPU** — usar CPU runtime para no gastar unidades.

## 0 — Setup

In [ ]:
!cd /content/meli-contact-detection && git pull 2>/dev/null || \
  git clone https://github.com/ronald-uba/meli-contact-detection /content/meli-contact-detection

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
exec(open('/content/drive/MyDrive/contact-detection/scripts/colab_setup.py').read())

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'datasets', 'Pillow', 'requests'], check=True)
print('✅ deps OK')

In [ ]:
import sys, yaml
from pathlib import Path

REPO_DIR   = '/content/meli-contact-detection'
DRIVE_ROOT = '/content/drive/MyDrive/contact-detection'
SPLITS_DIR = f'{DRIVE_ROOT}/data/splits'
IMGS_DIR   = f'{DRIVE_ROOT}/data/images'

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

with open(f'{REPO_DIR}/configs/qwen25_3b.yaml') as f:
    cfg = yaml.safe_load(f)

MAX_IMAGES   = cfg['data']['max_images']
IMG_MAX_SIDE = cfg['data']['img_max_side']
PROMPT_CHARS = cfg['data']['prompt_max_chars']
SEED         = cfg['data']['seed']

print(f'max_images   : {MAX_IMAGES}')
print(f'img_max_side : {IMG_MAX_SIDE}')
print(f'splits dir   : {SPLITS_DIR}')
print(f'images dir   : {IMGS_DIR}')

## 1 — Descarga train (65k)

In [ ]:
from src.data.dataset import csv_to_dataset

print('Descargando imágenes de train (puede tardar 2-4h)...')
ds_train = csv_to_dataset(
    csv_path           = f'{SPLITS_DIR}/train.csv',
    img_dir            = f'{IMGS_DIR}/train',
    max_images         = MAX_IMAGES,
    img_max_side       = IMG_MAX_SIDE,
    prompt_max_chars   = PROMPT_CHARS,
    seed               = SEED,
    limit              = None,   # dataset completo
    n_download_workers = 16,
)
print(f'✅ train descargado: {len(ds_train):,} ejemplos')

## 2 — Descarga val (23k)

In [ ]:
print('Descargando imágenes de val...')
ds_val = csv_to_dataset(
    csv_path           = f'{SPLITS_DIR}/val.csv',
    img_dir            = f'{IMGS_DIR}/val',
    max_images         = MAX_IMAGES,
    img_max_side       = IMG_MAX_SIDE,
    prompt_max_chars   = PROMPT_CHARS,
    seed               = SEED,
    limit              = None,
    n_download_workers = 16,
)
print(f'✅ val descargado: {len(ds_val):,} ejemplos')

## 3 — Verificación

In [ ]:
import os

def count_images(split_dir):
    p = Path(split_dir)
    if not p.exists():
        return 0
    return sum(1 for _ in p.rglob('*.jpg'))

n_train_imgs = count_images(f'{IMGS_DIR}/train')
n_val_imgs   = count_images(f'{IMGS_DIR}/val')

missing_train = sum(1 for ex in ds_train for p in ex['image_path'] if not os.path.exists(p))
missing_val   = sum(1 for ex in ds_val   for p in ex['image_path'] if not os.path.exists(p))

print(f'Imágenes en Drive/train : {n_train_imgs:,}')
print(f'Imágenes en Drive/val   : {n_val_imgs:,}')
print(f'Paths faltantes train   : {missing_train}')
print(f'Paths faltantes val     : {missing_val}')

if missing_train + missing_val == 0:
    print('✅ Todas las imágenes están en Drive. Listo para entrenar.')
else:
    pct = (missing_train + missing_val) / (len(ds_train) + len(ds_val)) * 100
    print(f'⚠️  {missing_train + missing_val} paths faltantes ({pct:.1f}%) — re-correr las celdas de descarga.')